# Test on Unseen Data (New Environment)

**Purpose:** Evaluate the trained models (XGBoost, SVM, PyTorch) on a completely **unseen test set** collected from a different environment to assess generalization.

| Model | Accuracy | Class 1 Recall | TP | FN |
|-------|----------|----------------|----|----|
| XGBoost | 73.7% | 78.4% | 69 | 19 |
| SVM | 69.8% | 77.3% | 68 | 20 |
| PyTorch | N/A (load error) | — | — | — |

> **Note:** The PyTorch model could not be loaded due to a `torch` version mismatch (`module 'torch' has no attribute '_utils'`). This is a known issue when the model was saved with a different PyTorch version. Re-training or matching the PyTorch version resolves it.

---

## Cell 1 — Imports & Function Definitions (New Label Format)

In [ ]:
# ==========================================
# Cell 1: Imports and function definitions (new label format)
# ==========================================
import os
import re
import librosa
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Path configuration
# =========================
# Path to the NEW test data folder
TEST_DATA_FOLDER = r"E:\12345678"

# Paths to model and scaler files (same as before)
SCALER_PATH = r"E:\audio_scaler.pkl"
MODEL_XGB_PATH = r"E:\xgboost_audio_model.pkl"
MODEL_SVM_PATH = r"E:\svm_audio_model.pkl"
MODEL_DL_PATH = r"E:\dl_audio_model.pth"

# Audio parameters
SR = 22050
N_MFCC = 20

# =========================
# Label extraction function (corrected for new format)
# Format: ClassNumber_RandomStuff.wav  (e.g. 23_1.wav)
# =========================
def extract_label(filename):
    name = os.path.splitext(filename)[0]
    try:
        label_part = name.split('_')[0]
        return int(label_part)
    except:
        return None

# =========================
# Feature extraction function (unchanged)
# =========================
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=SR)

    if len(y) == 0:
        return None

    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val

    y, _ = librosa.effects.trim(y, top_db=20)

    features = []

    # ---- MFCC + Delta ----
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    for feat in (mfcc, delta, delta2):
        features.extend(np.mean(feat, axis=1))
        features.extend(np.std(feat, axis=1))

    # ---- Spectral Features ----
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y)[0]

    for feat in (centroid, bandwidth, rolloff, zcr, rms):
        features.append(np.mean(feat))
        features.append(np.std(feat))

    # ---- Doppler / spectral behavior ----
    slope = np.polyfit(range(len(centroid)), centroid, 1)[0]
    centroid_diff_std = np.std(np.diff(centroid))

    features.append(slope)
    features.append(centroid_diff_std)

    return np.array(features)

print("Functions defined successfully (New Label Format).")

## Cell 2 — Load Scaler & Process Test Data

In [ ]:
# ==========================================
# Cell 2: Load scaler and process new test data
# ==========================================

# 1. Load the scaler
print(f"Loading Scaler from: {SCALER_PATH}")
scaler = joblib.load(SCALER_PATH)

# 2. Read test files
print(f"Scanning folder: {TEST_DATA_FOLDER}")
files = [f for f in os.listdir(TEST_DATA_FOLDER) if f.endswith(".wav")]
print(f"Found {len(files)} files.")

# 3. Extract features
X_test = []
y_test = []

print("Extracting features (this may take a while)...")
for file in tqdm(files):
    label = extract_label(file)
    if label is None:
        continue

    path = os.path.join(TEST_DATA_FOLDER, file)
    feats = extract_features(path)

    if feats is not None:
        X_test.append(feats)
        y_test.append(label)

X_test = np.array(X_test)
y_test = np.array(y_test)

print(f"\nProcessed {len(X_test)} valid samples.")
print(f"Feature shape: {X_test.shape}")
print(f"Unique Labels found: {np.unique(y_test)}")

# 4. Scale the test data
X_test_scaled = scaler.transform(X_test)
print("Data scaled using the training scaler.")

## Cell 3 — Define Architecture & Load Models

In [ ]:
# ==========================================
# Cell 3: Define architecture and load models
# ==========================================
import torch
import torch.nn as nn

# PyTorch architecture definition
class AudioClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(AudioClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.layer2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)

        self.layer3 = nn.Linear(64, 32)

        self.output = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)

        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)

        x = self.layer3(x)
        x = self.relu(x)

        x = self.output(x)
        return x

# Load models
print(f"Loading XGBoost from: {MODEL_XGB_PATH}")
model_xgb = joblib.load(MODEL_XGB_PATH)

print(f"Loading SVM from: {MODEL_SVM_PATH}")
model_svm = joblib.load(MODEL_SVM_PATH)

# Attempt to load PyTorch
model_pt = None
try:
    print(f"Loading PyTorch Model from: {MODEL_DL_PATH}")
    num_classes = len(np.unique(y_test))
    input_dim = X_test_scaled.shape[1]
    device = torch.device("cpu")
    model_pt = AudioClassifier(input_dim=input_dim, num_classes=num_classes).to(device)
    model_pt.load_state_dict(torch.load(MODEL_DL_PATH, map_location=device))
    model_pt.eval()
    print("PyTorch Model Loaded.")
except Exception as e:
    print("WARNING: Could not load PyTorch model.")
    print(f"Error: {e}")
    model_pt = None

## Cell 4 — Run Predictions & Generate Reports

In [ ]:
# ==========================================
# Cell 4: Run predictions and generate reports
# ==========================================

results = []

# --- XGBoost ---
print("\n" + "="*60)
print("Evaluating XGBoost on Folder 12345678")
print("="*60)
y_pred_xgb = model_xgb.predict(X_test_scaled)
print(classification_report(y_test, y_pred_xgb, digits=3))
results.append({"Model": "XGBoost", "Accuracy": accuracy_score(y_test, y_pred_xgb), "Recall_C1": recall_score(y_test, y_pred_xgb, labels=[1], average='macro')})

# --- SVM ---
print("\n" + "="*60)
print("Evaluating SVM on Folder 12345678")
print("="*60)
y_pred_svm = model_svm.predict(X_test_scaled)
print(classification_report(y_test, y_pred_svm, digits=3))
results.append({"Model": "SVM", "Accuracy": accuracy_score(y_test, y_pred_svm), "Recall_C1": recall_score(y_test, y_pred_svm, labels=[1], average='macro')})

# --- PyTorch ---
if model_pt is not None:
    print("\n" + "="*60)
    print("Evaluating PyTorch DL on Folder 12345678")
    print("="*60)
    X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
    with torch.no_grad():
        logits = model_pt(X_test_tensor)
        y_pred_pt = torch.argmax(logits, dim=1).cpu().numpy()
    print(classification_report(y_test, y_pred_pt, digits=3))
    results.append({"Model": "PyTorch DL", "Accuracy": accuracy_score(y_test, y_pred_pt), "Recall_C1": recall_score(y_test, y_pred_pt, labels=[1], average='macro')})
else:
    print("\nSkipping PyTorch.")

## Cell 5 — Confusion Matrices (XGBoost vs SVM)

In [ ]:
# ==========================================
# Cell 5: Plot confusion matrices (XGBoost vs SVM)
# ==========================================

# Define class names for readability
class_names = ['0 (Far-App)', '1 (Danger)', '2 (Near-Rec)', '3 (Far-Rec)', '4 (Safe)']

# Compute matrices
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_svm = confusion_matrix(y_test, y_pred_svm)

# Create figure with two panels (XGBoost left, SVM right)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- XGBoost ---
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], linewidths=0.5, linecolor='gray')
axes[0].set_title('Confusion Matrix: XGBoost (Test Set 2)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

# --- SVM ---
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], linewidths=0.5, linecolor='gray')
axes[1].set_title('Confusion Matrix: SVM (Test Set 2)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.show()

# ==========================================
# Quick numeric analysis
# ==========================================
print("Confusion Matrix Analysis (XGBoost):")
print("-" * 50)
fn_c1_xgb = cm_xgb[1, 0] + cm_xgb[1, 2] + cm_xgb[1, 3] + cm_xgb[1, 4]
tp_c1_xgb = cm_xgb[1, 1]
print(f"Class 1 (Danger): TP={tp_c1_xgb}, FN (Missed)={fn_c1_xgb}")

print("\nConfusion Matrix Analysis (SVM):")
print("-" * 50)
fn_c1_svm = cm_svm[1, 0] + cm_svm[1, 2] + cm_svm[1, 3] + cm_svm[1, 4]
tp_c1_svm = cm_svm[1, 1]
print(f"Class 1 (Danger): TP={tp_c1_svm}, FN (Missed)={fn_c1_svm}")